# ✋ See, Detect, Understand: Build a Hand Gesture Recogniser

## INSTRUCTOR TEMPLATE

In this hands-on, we will use **MediaPipe** to extract hand landmarks and train our own gesture classifier.

**Our challenge:**

> Can we build a system that recognises hand gestures from images and live webcam input?

### Learning Goals

By the end of this hands-on, you should be able to:

- Extract and visualise hand landmarks using MediaPipe.
- Convert landmarks into wrist-relative, scale-normalised features.
- Build a labelled gesture dataset and train a classifier.
- Evaluate predictions and test the complete pipeline using a webcam.

## 0. Setup

Run the following cells to prepare the environment and initialise MediaPipe.

### 0.1 Install dependencies

In [ ]:
# If you have not installed the required packages, uncomment and run:
# %pip install -q -r requirements.txt

# Note: VS Code users must install ipykernel in their selected
# environment before running notebook cells.

### 0.2 Import Libraries

In [ ]:
import os
import logging
import warnings

# Reduce unnecessary logs and warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GLOG_minloglevel"] = "3"
os.environ["ABSL_MIN_LOG_LEVEL"] = "3"

warnings.filterwarnings("ignore")
logging.getLogger("mediapipe").setLevel(logging.ERROR)
logging.getLogger("absl").setLevel(logging.ERROR)

from pathlib import Path
from urllib.request import urlretrieve

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

print("Setup successful!")

### 0.3 Download MediaPipe model

This is a pretrained MediaPipe Hand Landmarker model itself.

In [ ]:
MODEL_PATH = Path("hand_landmarker.task")

MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/"
    "hand_landmarker.task"
)

if not MODEL_PATH.exists():
    print("Downloading MediaPipe Hand Landmarker model...")
    urlretrieve(MODEL_URL, MODEL_PATH)

print("Model ready:", MODEL_PATH.exists())

**[INSTRUCTOR NOTE]**

MediaPipe also provides a pretrained Gesture Recognizer that can classify common hand gestures directly.

We use the Hand Landmarker instead because participants will train their own gesture classifier using the extracted landmarks.

### 0.4 Initialise the Hand Landmarker

In [ ]:
base_options = python.BaseOptions(
    model_asset_path=str(MODEL_PATH)
)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=1
)

landmarker = vision.HandLandmarker.create_from_options(options)

print("Hand Landmarker initialised!")

## 1. Our Gesture Recognition Task ✋

We will use **MediaPipe** to extract hand landmarks and train a classifier to recognise four hand gestures.

We will use a subset of the **HaGRID dataset**, containing 400 images across four gesture classes.

| Gesture | Label | Images |
|---|---|---|
| ✊ Fist | `fist` | 100 |
| 👍 Thumbs Up | `like` | 100 |
| ✋ Open Palm | `palm` | 100 |
| ✌️ Peace | `peace` | 100 |

Our goal is to convert these images into numerical landmark features, train a classifier and test it using live webcam input.

## 2. Explore Hand Landmarks

Let's use MediaPipe to detect and visualise the 21 hand landmarks in a HaGRID image.

### 2.1 Load and Detect Hand Landmarks

Let's start with an open palm image and extract its landmarks using MediaPipe.

In [ ]:
IMAGE_PATH = sorted(Path("data/hagrid/palm").glob("*.jpg"))[0]

image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise FileNotFoundError(IMAGE_PATH)

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=np.ascontiguousarray(image_rgb)
)

result = landmarker.detect(mp_image)

if not result.hand_landmarks:
    raise ValueError("No hand detected. Try another image.")

hand_landmarks = result.hand_landmarks[0]

print("Landmarks detected:", len(hand_landmarks))

### 2.2 Visualise the 21 Landmarks

Let's visualise the detected landmarks and their connections on the original image.

In [ ]:
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17)
]

def draw_hand_landmarks(image, landmarks):
    output = image.copy()
    h, w = output.shape[:2]

    points = [
        (int(lm.x * w), int(lm.y * h))
        for lm in landmarks
    ]

    for start, end in HAND_CONNECTIONS:
        cv2.line(output, points[start], points[end], (255, 255, 255), 2)

    for i, point in enumerate(points):
        cv2.circle(output, point, 5, (0, 255, 0), -1)
        cv2.putText(
            output, str(i), (point[0] + 5, point[1] - 5),
            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1
        )

    return output

In [ ]:
landmark_image = draw_hand_landmarks(image_rgb, hand_landmarks)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].imshow(image_rgb)
axes[0].set_title("Original Image")

axes[1].imshow(landmark_image)
axes[1].set_title("21 Hand Landmarks")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

### 2.3 Bounding Box and Zoom

Using the detected landmarks, we can calculate a bounding box around the hand and zoom in for a closer look.

In [ ]:
# Convert landmarks to pixel coordinates
h, w = image_rgb.shape[:2]

points = np.array([
    [int(lm.x * w), int(lm.y * h)]
    for lm in hand_landmarks
])

# Calculate bounding box with padding
padding = 30

x_min = max(0, points[:, 0].min() - padding)
y_min = max(0, points[:, 1].min() - padding)
x_max = min(w, points[:, 0].max() + padding)
y_max = min(h, points[:, 1].max() + padding)

# Draw bounding box on original image
boxed_image = image_rgb.copy()

cv2.rectangle(
    boxed_image,
    (x_min, y_min),
    (x_max, y_max),
    (255, 0, 0),
    3
)

# Crop the original image (without annotations)
cropped_hand = image_rgb[y_min:y_max, x_min:x_max].copy()

# Adjust landmark coordinates relative to the crop
crop_points = points - [x_min, y_min]

# Draw connections
for start, end in HAND_CONNECTIONS:
    cv2.line(
        cropped_hand,
        tuple(crop_points[start]),
        tuple(crop_points[end]),
        (255, 255, 255),
        1
    )

# Draw smaller landmarks without numbers
for point in crop_points:
    cv2.circle(
        cropped_hand,
        tuple(point),
        2,
        (0, 255, 0),
        -1
    )

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].imshow(boxed_image)
axes[0].set_title("Hand Bounding Box")

axes[1].imshow(cropped_hand)
axes[1].set_title("Zoomed Hand Landmarks")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

**Observe:** The bounding box is calculated from the detected landmarks, while the zoomed image allows us to inspect the hand structure more closely.

### 💻 Exercise 1: Inspect Hand Landmarks

Recall that each landmark contains `(x, y, z)` coordinates:

- **x, y:** Normalised position within the image.
- **z:** Relative depth, with the wrist as the origin.

Retrieve the coordinates of the **index fingertip (landmark 8)** and **wrist (landmark 0)**.

What do you notice about their coordinates?

In [ ]:
# TODO: Select the index fingertip and wrist
# index_tip = hand_landmarks[____]
# wrist = hand_landmarks[____]

index_tip = hand_landmarks[8]
wrist = hand_landmarks[0]

print("Index fingertip:")
print(f"x={index_tip.x:.3f}, y={index_tip.y:.3f}, z={index_tip.z:.3f}")

print("\nWrist:")
print(f"x={wrist.x:.3f}, y={wrist.y:.3f}, z={wrist.z:.3f}")

**Observe:** What do you notice about the wrist's `(x, y, z)` coordinates?

What happens to these coordinates if we move the hand without changing the gesture?

The wrist's x and y coordinates describe its position in the image, while z is approximately zero.

## 3. From Landmarks to Gesture Features

We now have 21 hand landmarks, each containing `(x, y, z)` coordinates.

This gives us **21 × 3 = 63 numerical values**.

But are raw landmark coordinates the best representation for recognising gestures?

### 3.1 A Problem With Raw Coordinates

Imagine showing the same ✌️ gesture at two different positions in an image.

Although the gesture remains the same, its landmark coordinates change.

**Our challenge:** How can we represent the hand's structure while reducing the influence of its position?

In [ ]:
# Simulate moving the same hand horizontally
original_x = hand_landmarks[8].x
shifted_x = original_x + 0.2

print(f"Original index fingertip x: {original_x:.3f}")
print(f"Shifted index fingertip x:  {shifted_x:.3f}")

*For demonstration only.*

The fingertip's position changed, but the gesture did not.

Instead of using absolute positions, let's represent landmarks relative to the wrist.

### 💻 Exercise 2: Wrist-Relative Coordinates

To reduce the effect of hand position, subtract the **wrist coordinates (landmark 0)** from each landmark.

$P_{\text{relative}}=P_{\text{landmark}}-P_{\text{wrist}}$

**Your task:** Calculate the index fingertip's `(x, y, z)` coordinates relative to the wrist.

In [ ]:
wrist = hand_landmarks[0]

# TODO: Calculate wrist-relative coordinates
# relative_x = __
# relative_y = __
# relative_z = __

relative_x = hand_landmarks[8].x - wrist.x
relative_y = hand_landmarks[8].y - wrist.y
relative_z = hand_landmarks[8].z - wrist.z

print("Relative index fingertip:", relative_x, relative_y, relative_z)

### 3.2 Scale Normalisation

Wrist-relative coordinates reduce the effect of hand position, but not size.

We also divide the coordinates by the **maximum distance from the wrist** to reduce differences in hand scale.

$P_{\text{normalised}}=\frac{P_{\text{relative}}}{\max_i\|P_{\text{relative},i}\|}$

Let's combine both transformations into one reusable function.

In [ ]:
def create_features(hand_landmarks):
    wrist = hand_landmarks[0]
    features = []

    for landmark in hand_landmarks:

        # Calculate wrist-relative coordinates
        relative_x = landmark.x - wrist.x
        relative_y = landmark.y - wrist.y
        relative_z = landmark.z - wrist.z

        features.extend([relative_x, relative_y, relative_z])

    # Convert to (21, 3) array
    points = np.array(features, dtype=np.float32).reshape(21, 3)

    # Normalise by maximum distance from wrist
    scale = np.max(np.linalg.norm(points, axis=1))

    if scale < 1e-6:
        return None

    points /= scale

    return points.flatten()

### 3.3 Inspect the Features

Let's check that our function produces 63 normalised features.

In [ ]:
features = create_features(hand_landmarks)

assert features is not None
assert len(features) == 63
assert np.allclose(features[:3], [0, 0, 0])

print("Number of features:", len(features))
print("Wrist coordinates:", features[:3])

We now have **63 wrist-relative, scale-normalised features** for each detected hand.

Next, we'll extract these features from our HaGRID images to build the training dataset.

## 4. Build Our Gesture Dataset

Let's apply our `create_features()` function to all 400 HaGRID images.

For each image, we will extract 63 normalised landmark features and pair them with the correct gesture label.

Images where MediaPipe cannot detect a hand will be skipped.

### 4.1 Load the Gesture Images

Let's check the number of images in each gesture class.

In [ ]:
DATA_DIR = Path("data/hagrid")
GESTURES = ["fist", "like", "palm", "peace"]

image_paths = {
    gesture: sorted(
        p for p in (DATA_DIR / gesture).rglob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    for gesture in GESTURES
}

for gesture, paths in image_paths.items():
    print(f"{gesture}: {len(paths)} images")

### 4.2 Extract Features from All Images

Let's process each image using MediaPipe and store its features and gesture label.

In [ ]:
X = []
y = []
image_files = []
failed_images = []

for gesture, paths in image_paths.items():
    for path in paths:
        image = cv2.imread(str(path))

        if image is None:
            failed_images.append(str(path))
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=image_rgb
        )

        result = landmarker.detect(mp_image)

        if not result.hand_landmarks:
            failed_images.append(str(path))
            continue

        features = create_features(result.hand_landmarks[0])

        if features is None:
            failed_images.append(str(path))
            continue

        X.append(features)
        y.append(gesture)
        image_files.append(str(path))

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("Feature matrix:", X.shape)
print("Labels:", y.shape)
print("Skipped images:", len(failed_images))

### 4.3 Inspect the Dataset

Let's check the resulting dataset and class distribution.

In [ ]:
df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(63)])
df["gesture"] = y
df["image"] = image_files

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["gesture"].value_counts())

### 4.4 Prepare Training and Test Sets

Split the dataset into **80% training** and **20% testing**, maintaining similar proportions of each gesture class.

In [ ]:
indices = np.arange(len(y))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

## 5. Train the Gesture Classifier

Now, let's train a **Random Forest classifier** using the 63 normalised landmark features extracted from our images.

Our classifier learns from numerical features, **not the original images**.

### 💻 Exercise 3: Train Our Gesture Classifier

**Your task:** Train the classifier using `X_train` and `y_train`.

**Hint:** Use the `.fit()` method.

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# TODO: Train the classifier
# model.fit(____, ____)
model.fit(X_train, y_train)

if not hasattr(model, "classes_"):
    raise NotImplementedError("Complete Exercise 3 before continuing.")

print("Gesture classifier trained successfully!")

## 6. Evaluate the Gesture Classifier

A high confidence score for one prediction does not necessarily mean a model performs well overall.

Let's evaluate our classifier using the **20% of HaGRID images held out during training**.

### 6.1 Evaluate Model Accuracy

**Accuracy** measures the proportion of correctly classified images.

$\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}}$

In [ ]:
# Predict gestures on the test set
predictions = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, predictions)

correct = np.sum(y_test == predictions)
total = len(y_test)

print(f"Correct predictions: {correct}/{total}")
print(f"Test accuracy: {accuracy:.1%}")

### 6.2 Which Gestures Were Confused?

Accuracy tells us how often the classifier is correct, but not which gestures it confuses.

Let's inspect the **confusion matrix**.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    labels=model.classes_,
    cmap="Blues",
    values_format="d"
)

plt.title("Gesture Recognition: Test Set")
plt.tight_layout()
plt.show()

### 6.3 Quick Discussion 🔍

- Which gestures were confused with one another?
- Does high test accuracy guarantee good performance on a new person's hand?

**Remember:** Our test set contains held-out images, but may include people already represented in the training set.

**[INSTRUCTOR NOTE]**

Classification accuracy excludes images where MediaPipe failed to detect a hand. Also, held-out images do not necessarily represent unseen people or webcam conditions.

## 7. Test the Full CV Pipeline

Let's apply our trained classifier to a held-out image and live webcam input.

**Our pipeline:**

Image → MediaPipe → Landmarks → Normalised Features → Gesture Prediction

### 7.1 Predict a Gesture from a Held-Out Image

Let's test our complete pipeline on an image from the held-out test set.

Compare the predicted gesture with its actual label.

In [ ]:
# Select a held-out test image
sample_idx = 0
UNSEEN_IMAGE_PATH = image_files[test_idx[sample_idx]]

# Load image
image = cv2.imread(UNSEEN_IMAGE_PATH)

if image is None:
    raise FileNotFoundError(UNSEEN_IMAGE_PATH)

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Detect hand landmarks
mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=np.ascontiguousarray(image_rgb)
)

result = landmarker.detect(mp_image)

if not result.hand_landmarks:
    raise ValueError("No hand detected.")

landmarks = result.hand_landmarks[0]

# Extract features and predict
features = create_features(landmarks)

if features is None:
    raise ValueError("Invalid landmark features.")

X_unseen = features.reshape(1, -1)

prediction = model.predict(X_unseen)[0]
probabilities = model.predict_proba(X_unseen)[0]

# Visualise image and landmarks
landmark_image = draw_hand_landmarks(image_rgb, landmarks)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(image_rgb)
axes[0].set_title("Original Image")

axes[1].imshow(landmark_image)
axes[1].set_title("Detected Landmarks")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

# Display prediction
print("Actual gesture:", y_test[sample_idx])
print("Predicted gesture:", prediction)
print("\nPrediction scores:")

for label, probability in zip(model.classes_, probabilities):
    print(f"{label:8s}: {probability:.1%}")

**Try it yourself:** Change `sample_idx` to test another held-out image. Does the prediction change?

### 7.2 Live Webcam Gesture Recognition 📷

Let's use our trained classifier to recognise gestures through a webcam!

Try showing `fist`, `like`, `palm` and `peace`.

Observe the predicted gesture and confidence score.

**Press Q to stop the webcam.**

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    cap.release()
    raise RuntimeError("Could not access webcam.")

print("Webcam started! Press 'q' to quit.")

try:
    while True:
        success, frame = cap.read()

        if not success:
            break

        # Convert BGR to RGB for MediaPipe
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=np.ascontiguousarray(frame_rgb)
        )

        # Detect hand landmarks
        result = landmarker.detect(mp_image)

        label = "No hand detected"
        prediction_score = None

        if result.hand_landmarks:
            landmarks = result.hand_landmarks[0]

            # Extract the same 63 features used during training
            features = create_features(landmarks)

            if features is not None:
                X_live = features.reshape(1, -1)

                # Predict using our trained Random Forest
                label = model.predict(X_live)[0]
                prediction_score = model.predict_proba(X_live).max()

            # Draw hand landmarks
            frame_rgb = draw_hand_landmarks(
                frame_rgb, landmarks
            )

        # Convert back to BGR for OpenCV display
        frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        # Display predicted gesture
        cv2.putText(
            frame,
            f"Gesture: {label}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        # Display prediction score
        if prediction_score is not None:
            cv2.putText(
                frame,
                f"Prediction Score: {prediction_score:.1%}",
                (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

        cv2.imshow("Live Gesture Recognition", frame)

        # Press Q to quit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()

print("Webcam stopped.")

### 7.3 Final Reflection 🔍

- Does the model recognise all four gestures correctly?
- What happens when you change your hand's distance or orientation?

**Key takeaway:** Good test accuracy does not guarantee reliable real-world performance.

### Instructor Note: Model Limitations and Generalisation

**Q: Why does the model sometimes predict incorrectly when the hand is rotated or viewed from a different angle?**

**A:** Our classifier learns patterns from the hand landmarks extracted by MediaPipe. If the training dataset contains mostly similar hand orientations, the model may struggle to recognise gestures from unfamiliar angles.

This is an example of a **generalisation limitation**. Improving dataset diversity, such as including different hand orientations and positions, can help the model perform better on unseen examples.

**Note:** If MediaPipe successfully detects the hand but the classifier predicts the wrong gesture, the issue may lie in the training data or feature representation rather than hand detection itself.